In [1]:
# Bounded Kaggle Qwen2.5-Coder-14B-Instruct 4-bit evaluation
# Private notebook; no public HTTP listener, no production access, no tunnel credentials.
import os, time, json, subprocess, sys
assert os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None or os.path.exists("/kaggle"), "Kaggle runtime required"
r=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.free","--format=csv,noheader"],capture_output=True,text=True,timeout=10)
print("GPU_STATUS",r.returncode,r.stdout[:400])
assert r.returncode==0, "GPU unavailable"


GPU_STATUS 0 Tesla T4, 15360 MiB, 14912 MiB
Tesla T4, 15360 MiB, 14912 MiB



In [2]:
# Packages installed only within the disposable Kaggle notebook runtime.
subprocess.run([sys.executable,"-m","pip","-q","install","-U","transformers>=4.48,<5","accelerate>=1.0,<2","bitsandbytes>=0.45,<1"],check=True,timeout=240)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', '-q', 'install', '-U', 'transformers>=4.48,<5', 'accelerate>=1.0,<2', 'bitsandbytes>=0.45,<1'], returncode=0)

In [ ]:
# Verify model loading first: 4-bit single-GPU only; no automatic multi-GPU fallback.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
model_id="Qwen/Qwen2.5-Coder-14B-Instruct"
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
start=time.monotonic()
tokenizer=AutoTokenizer.from_pretrained(model_id,trust_remote_code=False)
model=AutoModelForCausalLM.from_pretrained(model_id,quantization_config=quant,device_map={"":0},torch_dtype=torch.float16,low_cpu_mem_usage=True,trust_remote_code=False)
model.eval()
print("MODEL_LOADED",round(time.monotonic()-start,2),"seconds")


In [12]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("KAGGLE_INFERENCE_BEARER")


In [ ]:
# Local assertions; never expose an inference server or run generated commands.
import re
tasks=[
 {"id":"ts-undefined","prompt":"Fix TypeScript strict noUncheckedIndexedAccess: function first<T>(xs:T[]):T{return xs[0]}. Return only the corrected function."},
 {"id":"ci-lockfile","prompt":"For pnpm GitHub Actions CI, identify the correct lockfile and frozen install command. Two short lines."},
 {"id":"agent-json","prompt":"Return only minified JSON: max_tool_calls=3, stop_on_error=true, states=[plan,tool,done,failed]."},
]
def generate(prompt):
 messages=[{"role":"system","content":"You are a careful coding assistant. Never execute code or call tools."},{"role":"user","content":prompt}]
 text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
 inputs=tokenizer(text,return_tensors="pt",truncation=True,max_length=2048).to("cuda:0")
 start=time.monotonic()
 with torch.inference_mode(): output=model.generate(**inputs,max_new_tokens=144,do_sample=False,pad_token_id=tokenizer.eos_token_id)
 return tokenizer.decode(output[0][inputs.input_ids.shape[1]:],skip_special_tokens=True),round(time.monotonic()-start,2)
results=[]
for task in tasks:
 try:
  answer,latency=generate(task["prompt"])
  record={"task":task["id"],"latency_seconds":latency,"answer":answer[:2500]}
  if task["id"]=="ts-undefined":record["heuristic_pass"]=bool(re.search(r"T\s*\|\s*undefined|throw\s+new|xs\.length",answer))
  if task["id"]=="ci-lockfile":record["heuristic_pass"]="pnpm-lock.yaml" in answer and "--frozen-lockfile" in answer
  if task["id"]=="agent-json":
   try:
    j=json.loads(answer.strip());record["heuristic_pass"]=j=={"max_tool_calls":3,"stop_on_error":True,"states":["plan","tool","done","failed"]}
   except (ValueError,TypeError):record["heuristic_pass"]=False
  results.append(record);print("TASK",task["id"],"PASS",record["heuristic_pass"],"LATENCY",latency)
 except Exception as exc:
  results.append({"task":task["id"],"error":type(exc).__name__});print("TASK_ERROR",task["id"],type(exc).__name__)
with open("/kaggle/working/qwen25_bounded_results.json","w") as f:json.dump({"model":model_id,"quant":"nf4-4bit","results":results},f,indent=2)
print("BENCHMARK_FINISHED",len(results))


In [ ]:
# Tailscale private connection: userspace mode works without /dev/net/tun.
# Joins as an ephemeral Kaggle GPU peer; never creates a public endpoint.
import os, subprocess, tempfile, pathlib, time, shutil
from kaggle_secrets import UserSecretsClient
key = None
secret_error_class = None
try:
    key = UserSecretsClient().get_secret("TAILSCALE_AUTHKEY")
except Exception as exc:
    secret_error_class = type(exc).__name__
print("TAILSCALE_SECRET_PRESENT", bool(key))
if secret_error_class:
    print("TAILSCALE_SECRET_ERROR_CLASS", secret_error_class)
if not key:
    print("TAILSCALE_BLOCKED: secret unavailable in Kaggle runtime")
else:
    # Download the official static Tailscale binary into Kaggle ephemeral storage.
    import urllib.request, tarfile, io
    install_dir = pathlib.Path('/kaggle/working/tailscale-bin');install_dir.mkdir(exist_ok=True)
    release = '1.104.4'
    address = f'https://pkgs.tailscale.com/stable/tailscale_{release}_amd64.tgz'
    try:
        archive_path = install_dir / 'tailscale.tgz'
        with urllib.request.urlopen(address, timeout=60) as response:
            with archive_path.open('wb') as output:
                shutil.copyfileobj(response, output)
        print('TAILSCALE_DOWNLOAD_BYTES', archive_path.stat().st_size)
        with tarfile.open(archive_path, mode='r:gz') as package:
            root_prefix=f'tailscale_{release}_amd64/'
            for name in ('tailscale','tailscaled'):
                item=package.extractfile(root_prefix+name)
                if item is None: raise ValueError('Tailscale binary missing')
                out=install_dir/name
                with out.open('wb') as output:
                    shutil.copyfileobj(item, output)
                out.chmod(0o700)
        print('TAILSCALE_EXTRACTION_OK', True)
        socket='/kaggle/working/tailscaled.sock'
        daemon=subprocess.Popen([str(install_dir/'tailscaled'),'--tun=userspace-networking','--socket='+socket,'--state=mem:'],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
        time.sleep(2)
        # Ephemeral /dev/shm credential, removed even if the CLI fails.
        keyfile=pathlib.Path('/dev/shm/.tailscale-kaggle-authkey')
        keyfile.write_text(key);keyfile.chmod(0o600)
        try:
            run=subprocess.run([str(install_dir/'tailscale'),'--socket='+socket,'up','--auth-key=file:'+str(keyfile),'--hostname=kaggle-qwen25-coder','--accept-routes=false'],capture_output=True,text=True,timeout=35)
        finally:
            keyfile.unlink(missing_ok=True)
        if run.returncode==0:
            print('TAILSCALE_CONNECTED: yes')
            status=subprocess.run([str(install_dir/'tailscale'),'--socket='+socket,'status','--json'],capture_output=True,text=True,timeout=10)
            if status.returncode==0:
                data=__import__('json').loads(status.stdout)
                print('TAILSCALE_STATE',data.get('BackendState'))
                print('TAILSCALE_ADDR',((data.get('Self') or {}).get('TailscaleIPs') or [''])[0])
            ping=subprocess.run([str(install_dir/'tailscale'),'--socket='+socket,'ping','--c=1','100.65.237.22'],capture_output=True,text=True,timeout=15)
            print('ORACLE_TAILSCALE_PING',ping.returncode==0)
        else:
            print('TAILSCALE_JOIN_FAILED',run.returncode)
    except Exception as exc:
        print('TAILSCALE_SETUP_FAILED',type(exc).__name__)


In [ ]:
# Ephemeral authenticated API: local GPU check, then tailnet-only HTTPS Serve.
# The notebook imports a pinned source bundle from a PRIVATE Kaggle dataset;
# do not clone the private GitHub repository with embedded credentials.
from pathlib import Path
import sys, time
from kaggle_secrets import UserSecretsClient
source_root = Path('/kaggle/input/qwen25-private-inference-source')
if not (source_root / 'inference/kaggle_runtime.py').is_file():
    print('LIVE_API_BLOCKED_SOURCE_BUNDLE_MISSING')
else:
    sys.path.insert(0, str(source_root))
    from inference.kaggle_runtime import start_private_inference
    bearer = None
    try:
        bearer = UserSecretsClient().get_secret('KAGGLE_INFERENCE_BEARER')
    except Exception as exc:
        print('LIVE_API_SECRET_READ_ERROR', type(exc).__name__)
    if not isinstance(bearer, str) or len(bearer) < 32:
        print('LIVE_API_BLOCKED_INFERENCE_BEARER_MISSING_OR_SHORT')
    elif 'run' not in globals() or run.returncode != 0:
        print('LIVE_API_BLOCKED_TAILSCALE_NOT_CONNECTED')
    else:
        service = None
        try:
            service = start_private_inference(model, tokenizer, torch, bearer,
                str(install_dir / 'tailscale'), socket)
            print('LIVE_API_LOCAL_INFERENCE_AND_TAILNET_SERVE_READY')
            # Bounded opportunity for Oracle-side acceptance; then tear down.
            time.sleep(150)
        except Exception as exc:
            print('LIVE_API_TEST_FAILED', type(exc).__name__)
        finally:
            if service is not None:
                service.shutdown()
                service.server_close()
